# 🫀 실험 4 — 신호품질과 기권(abstain): **모를 때 모른다고 말하기**

**MedKOS / `notebooks/exp4_signal_quality_abstain.ipynb`** · 퀘스트 `ailab-2026-0015`

---

## 왜 이 실험인가

지금까지의 실험은 전부 **"맞히기"** 였습니다. 이건 **"틀릴 것 같으면 손 들기"** 입니다.

웨어러블 배치에서 가장 흔한 실패는 오진이 아니라 **노이즈를 병으로 읽는 것**입니다.
그래서 `ailab-2026-0016`에서 **abstain 헤드**("이 신호로는 판단 불가")를 설계에 넣었는데,
**그게 실제로 도움이 되는지는 아직 한 번도 재보지 않았습니다.**

## 이 실험만의 강점 — 유일하게 인과 실험이 된다

다른 축(RR·P파·ST-T)은 전부 **관찰 연구**입니다. 노이즈만은 다릅니다:

> **`nstdb`의 실제 노이즈(baseline wander · 근전도 · 전극 이동)를
> 원하는 SNR로 정확히 주입할 수 있습니다.**

원인을 우리가 조작하므로 **"노이즈가 성능을 얼마나 떨어뜨리는가"를 인과적으로** 잴 수 있습니다.
공개 ECG 연구에서 드문 조건입니다.

## 사전등록

```
Q1 (품질 추정)  : SNR을 신호만 보고 맞힐 수 있는가 → MAE(dB)
Q2 (성능 저하)  : SNR이 내려가면 분류 성능이 얼마나 떨어지는가 → SNR별 macro-F1 곡선
Q3 (기권 효용)  : 저품질을 기권시키면 남은 것의 성능이 오르는가  ★ 주가설
                  같은 커버리지에서 '품질 기반 기권' vs '확률 기반 기권' vs '무작위 기권'
지표   : coverage-accuracy 곡선의 AUC + 커버리지 80%에서의 macro-F1
판정   : 품질 기반 기권이 무작위 기권보다 유의하게 나은가 (부트스트랩 CI)
금지   : 테스트 SNR을 보고 임계값을 조정하지 않는다 (임계는 학습셋에서만)
```

**세 번째 대조군(무작위 기권)이 핵심**입니다. 기권하면 어려운 것도 같이 빠지므로
**아무 기준으로나 기권해도 성능은 오릅니다.** 무작위 대비 이득이 있어야 진짜입니다.


In [ ]:
# CELL 1 — 설정 + Drive lib
!pip -q install wfdb

import os, sys, json, time, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

FS, W_PRE, W_POST = 360, 144, 216          # mitdb 360Hz, R중심 1초
SNRS = [24, 18, 12, 6, 0, -6]              # nstdb 표준 SNR 계단
CLASSES = ["N", "S", "V"]
SEED0, N_SEEDS, EPOCHS = 20260731, 2, 15
QUICK = True

CONFIG = dict(exp="exp4_signal_quality_abstain", quest="ailab-2026-0015",
              hypothesis="품질 기반 기권이 무작위 기권보다 유의하게 나은가",
              causal="nstdb 노이즈를 알려진 SNR로 주입 — 원인을 조작하는 유일한 축",
              snrs=SNRS, fs=FS, seed0=SEED0, quick=QUICK, boot=2000)
np.random.seed(SEED0)
run = MedKOSRun("exp4_quality", CONFIG, project=PROJECT)

In [ ]:
# CELL 2 — mitdb 비트 + nstdb 노이즈 확보 (★ 구조 확인)
import wfdb
from collections import Counter

DS1 = [101,106,108,109,112,114,115,116,118,119,122,124,201,203,205,207,208,209,215,220,223,230]
DS2 = [100,103,105,111,113,117,121,123,200,202,210,212,213,214,219,221,222,228,231,232,233,234]
AAMI = {**{s: 0 for s in "NLRej"}, **{s: 1 for s in ["A","a","J","S"]},
        **{s: 2 for s in ["V","E"]}}          # F·Q 제외(이번 실험 초점 아님)

# 노이즈 레코드
NOISE = {}
for nm in ("bw", "em", "ma"):
    try:
        r = wfdb.rdrecord(nm, pn_dir="nstdb")
        NOISE[nm] = r.p_signal[:, 0].astype("float32")
        run.log(f"✅ nstdb/{nm}: fs={r.fs} len={len(NOISE[nm]):,} "
                f"std={NOISE[nm].std():.3f}")
    except Exception as e:
        run.log(f"❌ nstdb/{nm}: {type(e).__name__}: {str(e)[:80]}")
run.save_json("noise_probe", {k: {"len": int(len(v)), "std": float(v.std())}
                              for k, v in NOISE.items()})
if not NOISE:
    run.log("⛔ 노이즈 레코드를 못 받았습니다 — 여기서 멈추세요")

In [ ]:
# CELL 3 — 깨끗한 비트 수집 (mitdb)
def robust(x):
    q = np.percentile(x, 75) - np.percentile(x, 25)
    return ((x - np.median(x)) / (q + 1e-6)).astype("float32")

CACHE = run.data(f"mitdb_beats_{FS}hz.npz")
if os.path.exists(CACHE):
    z = np.load(CACHE); Xc, Yc, Pc = z["X"], z["y"], z["pid"]
    run.log(f"캐시 재사용 {Xc.shape}")
else:
    Xs, Ys, Ps = [], [], []
    for rid in DS1 + DS2:
        try:
            rec = wfdb.rdrecord(str(rid), pn_dir="mitdb")
            ann = wfdb.rdann(str(rid), "atr", pn_dir="mitdb")
        except Exception as e:
            run.log(f"  [skip {rid}] {str(e)[:60]}"); continue
        names = [n.strip() for n in rec.sig_name]
        ch = names.index("MLII") if "MLII" in names else 0
        sig = robust(rec.p_signal[:, ch].astype("float64"))
        for s_, sym in zip(ann.sample, ann.symbol):
            if sym not in AAMI or s_ - W_PRE < 0 or s_ + W_POST > len(sig):
                continue
            Xs.append(sig[s_ - W_PRE:s_ + W_POST]); Ys.append(AAMI[sym]); Ps.append(rid)
    Xc = np.array(Xs, "float32"); Yc = np.array(Ys); Pc = np.array(Ps)
    np.savez_compressed(CACHE, X=Xc, y=Yc, pid=Pc)
    run.log(f"저장 {CACHE}")
run.log(f"비트 {len(Yc):,} · 환자 {len(np.unique(Pc))} · 클래스 {np.bincount(Yc).tolist()}")

def add_noise(X, snr_db, rng):
    """알려진 SNR로 nstdb 실제 노이즈를 주입한다(합성 가우시안 아님)."""
    out = np.empty_like(X)
    keys = list(NOISE)
    for i in range(len(X)):
        nz = NOISE[keys[rng.randint(len(keys))]]
        st = rng.randint(0, len(nz) - X.shape[1])
        n = nz[st:st + X.shape[1]]
        n = n - n.mean()
        ps, pn = np.mean(X[i] ** 2), np.mean(n ** 2) + 1e-12
        out[i] = X[i] + n * np.sqrt(ps / (pn * 10 ** (snr_db / 10)))
    return out

In [ ]:
# CELL 4 — 분류기 + 품질(SNR) 추정기 학습
import tensorflow as tf
from tensorflow.keras import layers, models

tr_m, te_m = np.isin(Pc, DS1), np.isin(Pc, DS2)
rng = np.random.RandomState(SEED0)
if QUICK:                                   # N 언더샘플(실험1′과 동일 철학)
    idx = np.where(tr_m)[0]
    ect = idx[Yc[idx] != 0]; nrm = idx[Yc[idx] == 0]
    keep = min(len(nrm), len(ect) * 10)
    tr_idx = np.sort(np.concatenate([ect, rng.choice(nrm, keep, replace=False)]))
else:
    tr_idx = np.where(tr_m)[0]
te_idx = np.where(te_m)[0]
run.log(f"학습 {len(tr_idx):,} / 테스트 {len(te_idx):,}")

# 학습셋: 여러 SNR을 섞어 넣는다(품질에 강건한 분류기 + SNR 추정 라벨 확보)
Xtr_l, Ytr_l, Str_l = [], [], []
for snr in SNRS:
    Xtr_l.append(add_noise(Xc[tr_idx], snr, rng))
    Ytr_l.append(Yc[tr_idx]); Str_l.append(np.full(len(tr_idx), snr, "float32"))
Xtr = np.concatenate(Xtr_l)[..., None]
Ytr = np.concatenate(Ytr_l); Str = np.concatenate(Str_l)
run.log(f"증강 학습셋 {Xtr.shape} (SNR {SNRS} 각각)")

def backbone(seed, out_units, out_act, loss):
    tf.keras.utils.set_random_seed(seed)
    si = layers.Input((W_PRE + W_POST, 1)); x = si
    for f, k in ((32, 7), (64, 5), (128, 3)):
        x = layers.Conv1D(f, k, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x); x = layers.MaxPooling1D(2)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x); x = layers.Dropout(0.3)(x)
    m = models.Model(si, layers.Dense(out_units, activation=out_act)(x))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0), loss=loss)
    return m

def auto_weights(y, beta=0.9999):
    w = {c: (1 - beta) / (1 - beta ** max((y == c).sum(), 1)) for c in range(3)}
    return {c: float(v / w[0]) for c, v in w.items()}
CW = auto_weights(Ytr); SW = np.array([CW[int(c)] for c in Ytr], "float32")

CLF = backbone(SEED0, 3, "softmax", "sparse_categorical_crossentropy")
CLF.fit(Xtr, Ytr, epochs=EPOCHS, batch_size=256, sample_weight=SW, verbose=0)
run.log("분류기 학습 완료")

QUAL = backbone(SEED0 + 1, 1, "linear", "mse")     # Q1: SNR 회귀
QUAL.fit(Xtr, Str, epochs=EPOCHS, batch_size=256, verbose=0)
run.log("품질 추정기 학습 완료")
run.save_model(CLF, "classifier"); run.save_model(QUAL, "quality")

In [ ]:
# CELL 5 — Q1·Q2: SNR 추정 정확도 + SNR별 성능 저하 곡선
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

rng2 = np.random.RandomState(SEED0 + 7)
rows, per_snr = [], {}
for snr in SNRS:
    Xt = add_noise(Xc[te_idx], snr, rng2)[..., None]
    yhat = CLF.predict(Xt, batch_size=1024, verbose=0)
    shat = QUAL.predict(Xt, batch_size=1024, verbose=0).ravel()
    f1 = f1_score(Yc[te_idx], yhat.argmax(1), average="macro", zero_division=0)
    mae = float(np.mean(np.abs(shat - snr)))
    per_snr[snr] = {"macro_f1": float(f1), "snr_mae": mae,
                    "snr_pred_mean": float(shat.mean())}
    rows.append((snr, f1, mae, shat.mean()))
    run.log(f"  SNR {snr:+3d} dB → macro-F1 {f1:.4f} · SNR추정 평균 {shat.mean():+.1f} "
            f"(MAE {mae:.2f} dB)")

overall_mae = float(np.mean([v["snr_mae"] for v in per_snr.values()]))
run.log(f"\nQ1 SNR 추정 전체 MAE = {overall_mae:.2f} dB "
        f"({'✅ 쓸 만함' if overall_mae < 4 else '⚠️ 부정확 — 기권 기준으로 약함'})")
drop = per_snr[SNRS[0]]["macro_f1"] - per_snr[SNRS[-1]]["macro_f1"]
run.log(f"Q2 성능 저하 = {drop:+.4f} (SNR {SNRS[0]}dB → {SNRS[-1]}dB)")
run.save_json("q1_q2_snr_curve", {"per_snr": per_snr, "snr_mae": overall_mae,
                                  "f1_drop": drop})

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot([r[0] for r in rows], [r[1] for r in rows], "o-", color="#1c4780")
ax[0].set_xlabel("SNR (dB)"); ax[0].set_ylabel("macro-F1"); ax[0].invert_xaxis()
ax[0].set_title("Q2 — 노이즈가 성능을 얼마나 떨어뜨리나")
ax[1].plot([r[0] for r in rows], [r[3] for r in rows], "o-", color="#c9762b")
ax[1].plot(SNRS, SNRS, "--", color="#999", lw=1)
ax[1].set_xlabel("실제 SNR"); ax[1].set_ylabel("추정 SNR"); ax[1].invert_xaxis()
ax[1].set_title("Q1 — SNR을 맞힐 수 있나")
plt.tight_layout(); run.save_fig("q1_q2_snr", fig); plt.show()

In [ ]:
# CELL 6 — Q3(주가설): 기권 전략 3종 비교 — 품질 vs 확률 vs 무작위
# 테스트셋을 여러 SNR이 섞인 '현실적 혼합'으로 만든다.
rng3 = np.random.RandomState(SEED0 + 13)
mix_snr = rng3.choice(SNRS, size=len(te_idx))
Xmix = np.stack([add_noise(Xc[te_idx][i:i+1], mix_snr[i], rng3)[0]
                 for i in range(len(te_idx))])[..., None]
ymix = Yc[te_idx]
pr = CLF.predict(Xmix, batch_size=1024, verbose=0)
snr_hat = QUAL.predict(Xmix, batch_size=1024, verbose=0).ravel()
conf = pr.max(1)
pred = pr.argmax(1)
run.log(f"혼합 테스트셋 {len(ymix):,}비트 · SNR 분포 "
        f"{ {int(s): int((mix_snr==s).sum()) for s in SNRS} }")

STRAT = {"품질(추정 SNR)": snr_hat, "확률(최대 softmax)": conf,
         "무작위": rng3.rand(len(ymix))}
COVS = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]
curves = {}
for nm, score in STRAT.items():
    ys = []
    for cov in COVS:
        k = int(len(ymix) * cov)
        keep = np.argsort(-score)[:k]          # 점수 높은 순으로 남긴다
        ys.append(f1_score(ymix[keep], pred[keep], average="macro", zero_division=0))
    curves[nm] = ys
    run.log(f"  {nm:16s} " + "  ".join(f"{c:.0%}:{v:.3f}" for c, v in zip(COVS, ys)))

def boot_at(cov, sa, sb, B=CONFIG["boot"], seed=SEED0):
    rs = np.random.RandomState(seed); n = len(ymix); out = []
    for _ in range(B):
        i = rs.randint(0, n, n)
        d = []
        for s in (sa, sb):
            k = int(n * cov); keep = i[np.argsort(-s[i])[:k]]
            d.append(f1_score(ymix[keep], pred[keep], average="macro", zero_division=0))
        out.append(d[0] - d[1])
    out = np.array(out)
    return float(out.mean()), float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))

run.log(f"\n▶ 주가설 — 커버리지 80%에서 '품질 기반' vs '무작위'")
d, l, u = boot_at(0.8, STRAT["품질(추정 SNR)"], STRAT["무작위"])
sig_q = bool(l > 0)
run.log(f"   품질 − 무작위 : Δ={d:+.4f}  CI [{l:+.4f}, {u:+.4f}]  "
        f"{'유의 ★' if sig_q else '비유의'}")
d2, l2, u2 = boot_at(0.8, STRAT["품질(추정 SNR)"], STRAT["확률(최대 softmax)"])
run.log(f"   품질 − 확률   : Δ={d2:+.4f}  CI [{l2:+.4f}, {u2:+.4f}]  "
        f"{'유의 ★' if (l2>0 or u2<0) else '비유의'}")

if sig_q and d > 0:
    verdict = "확증 — 신호품질 기반 기권은 무작위 기권보다 낫다. abstain 헤드에 근거가 생김"
elif l2 > 0:
    verdict = "부분 확증 — 품질이 확률보다는 낫지만 무작위 대비는 미검출"
else:
    verdict = ("기각 — 품질 기반 기권의 고유 이득 없음. 기권은 '어려운 것도 같이 빼서' "
               "오르는 것이며 품질 신호 자체의 기여는 아니다")
run.log(f"   → {verdict}")

fig, ax = plt.subplots(figsize=(6, 3.6))
for nm, ys in curves.items():
    ax.plot([c * 100 for c in COVS], ys, "o-", label=nm)
ax.set_xlabel("커버리지 (%)"); ax.set_ylabel("macro-F1"); ax.invert_xaxis()
ax.set_title("Q3 — 기권 전략 비교 (오른쪽→왼쪽으로 갈수록 많이 기권)")
ax.legend(); plt.tight_layout(); run.save_fig("q3_abstain", fig); plt.show()

run.save_json("q3_abstain", {"coverages": COVS, "curves": curves,
                             "quality_minus_random@0.8": {"delta": d, "ci": [l, u]},
                             "quality_minus_conf@0.8": {"delta": d2, "ci": [l2, u2]},
                             "verdict": verdict})
result = {"week": 1, "exp_id": "exp4_quality", "quest": "ailab-2026-0015",
          "task": "신호품질 추정과 기권(abstain)의 효용", "split": "inter",
          "metric": "macro_f1", "value": round(curves["품질(추정 SNR)"][2], 4),
          "passed": bool(sig_q and d > 0), "date": time.strftime("%Y-%m-%d"),
          "snr_mae": overall_mae, "f1_drop_24_to_-6": drop,
          "quality_minus_random@0.8": {"delta": d, "ci": [l, u]},
          "verdict": verdict,
          "summary": f"SNR MAE {overall_mae:.1f}dB · F1 저하 {drop:+.3f} · "
                     f"품질−무작위@80% {d:+.4f} [{l:+.4f},{u:+.4f}] → {verdict.split(' —')[0]}"}
result = run.finish(result)
import shutil; shutil.copy(os.path.join(run.dir, "result.json"), "/content/result.json")
print(f"""
────────────────────────────────────────────────────────────────
  python pipelines/ingest_run.py --results result.json \\
      --notebook notebooks/exp4_signal_quality_abstain.ipynb \\
      --quest ailab-2026-0015 --step "exp4-signal-quality-abstain" \\
      --note "{result['summary']}"
────────────────────────────────────────────────────────────────""")

---

## 결과 읽는 법

| 품질 − 무작위 @80% | 뜻 | 다음 |
|---|---|---|
| **유의하게 양수** | **abstain 헤드에 실증 근거가 생김.** 웨어러블 배치의 안전장치가 정당화됨 | Finding Head에 `quality` 가지를 정식 입력으로 |
| 비유의 | 기권의 이득은 "어려운 것도 같이 빠져서"일 뿐 | 품질 추정을 개선하거나(다중 노이즈 유형 분리) 기권 자체를 재고 |
| Q1 MAE > 4 dB | 품질 추정 자체가 부정확 | 기권 기준으로 쓰기 전에 추정기부터 고칠 것 |

## 왜 무작위 대조군이 필수인가
**기권하면 성능은 거의 항상 오릅니다** — 어려운 샘플이 같이 빠지니까요.
그래서 "기권했더니 올랐다"는 아무 증거도 아닙니다.
**같은 커버리지에서 무작위보다 나아야** 품질 신호가 실제로 정보를 준 것입니다.
이 대조군이 없는 abstain 논문이 많은데, 그건 검증이 아닙니다.
